라이브러리 설치

In [6]:
!pip install -q transformers datasets scikit-learn pandas accelerate

2. GitHub 오픈소스 코드 수집 및 CSV 데이터셋 생성

In [16]:
import os
import pandas as pd

# =========================
# 2. GitHub 오픈소스 코드 수집 후 CSV 데이터셋 업데이트
# =========================

# CSV 파일 경로
csv_path = "github_vulnerability_dataset.csv"

# 수집할 GitHub 저장소 목록
# 새 오픈소스 저장소를 추가하고 싶으면 이 목록에만 추가하면 됨
REPOS = [
    {
        "url": "https://github.com/dehvCurtis/vulnerable-code-examples.git",
        "folder": "vulnerable-code-examples",
        "source": "dehvCurtis/vulnerable-code-examples"
    },
    {
        "url": "https://github.com/OWASP/Vulnerable-Web-Application.git",
        "folder": "OWASP-Vulnerable-Web-Application",
        "source": "OWASP/Vulnerable-Web-Application"
    } ,
     {
        "url": "https://github.com/npapernot/buffer-overflow-attack.git",
        "folder": "buffer-overflow-attack",
        "source": "npapernot/buffer-overflow-attack"
    },
    {
        "url": "https://github.com/wadejason/Buffer-Overflow-Vulnerability-Lab.git",
        "folder": "Buffer-Overflow-Vulnerability-Lab",
        "source": "wadejason/Buffer-Overflow-Vulnerability-Lab"
    }
]

# 수집할 코드 파일 확장자
CODE_EXTENSIONS = [".py", ".php", ".js", ".java", ".c", ".cpp", ".html"]


# =========================
# 파일 안전하게 읽는 함수
# =========================

def read_file_safely(file_path):
    encodings = ["utf-8", "latin-1", "cp949"]

    for enc in encodings:
        try:
            with open(file_path, "r", encoding=enc, errors="ignore") as f:
                return f.read()
        except:
            pass

    return None


# =========================
# 파일 경로와 코드 패턴을 기반으로 취약점 유형 추정
# =========================

def detect_vuln_type(file_path, code):
    path = file_path.lower()
    code_lower = code.lower()

    # SQL Injection
    if (
        "sqli" in path
        or "sql-injection" in path
        or "sql_injection" in path
        or ("select" in code_lower and ("$_get" in code_lower or "$_post" in code_lower))
    ):
        return "SQL_INJECTION"

    # XSS
    if (
        "xss" in path
        or "cross-site" in path
        or "innerhtml" in code_lower
        or "document.write" in code_lower
    ):
        return "XSS"

    # Command Injection
    if (
        "command" in path
        or "cmd" in path
        or "command-execution" in path
        or "os.system" in code_lower
        or "shell=true" in code_lower
        or "system(" in code_lower
    ):
        return "COMMAND_INJECTION"

    # Path Traversal
    if (
        "path-traversal" in path
        or "directory-traversal" in path
        or "traversal" in path
        or "lfi" in path
        or "../" in code_lower
        or "..\\" in code_lower
    ):
        return "PATH_TRAVERSAL"

    # Buffer Overflow
    if (
        "buffer" in path
        or "overflow" in path
        or "strcpy" in code_lower
        or "gets(" in code_lower
        or "memcpy" in code_lower
    ):
        return "BUFFER_OVERFLOW"

    return "UNKNOWN"


# =========================
# GitHub 저장소 clone
# =========================

for repo in REPOS:
    # 기존에 같은 폴더가 있으면 삭제
    !rm -rf {repo["folder"]}

    # GitHub 저장소 다운로드
    !git clone {repo["url"]} {repo["folder"]}

print("GitHub 저장소 다운로드 완료")


# =========================
# 코드 파일 수집
# =========================

rows = []

for repo in REPOS:
    base_dir = repo["folder"]

    for root, dirs, files in os.walk(base_dir):
        for file in files:
            ext = os.path.splitext(file)[1].lower()

            if ext in CODE_EXTENSIONS:
                file_path = os.path.join(root, file)
                code = read_file_safely(file_path)

                if code is None:
                    continue

                code = code.strip()

                # 너무 짧은 코드는 제외
                if len(code) < 30:
                    continue

                vuln_type = detect_vuln_type(file_path, code)

                # 취약점 유형을 알 수 없는 코드는 일단 제외
                if vuln_type == "UNKNOWN":
                    continue

                rows.append({
                    "code": code[:3000],
                    "vuln_type": vuln_type,
                    "source": repo["source"],
                    "file_path": file_path
                })

new_df = pd.DataFrame(rows)

print("새로 수집한 데이터 수:", len(new_df))

if len(new_df) > 0:
    print(new_df["vuln_type"].value_counts())
    display(new_df.head())
else:
    print("수집된 데이터가 없습니다.")


# =========================
# 기존 CSV 불러오기
# =========================

if os.path.exists(csv_path):
    old_df = pd.read_csv(csv_path)
    print("기존 CSV 데이터 수:", len(old_df))
else:
    old_df = pd.DataFrame(columns=["code", "vuln_type", "source", "file_path"])
    print("기존 CSV가 없어 새로 생성합니다.")


# =========================
# 기존 CSV + 새 데이터 합치기
# =========================

df = pd.concat([old_df, new_df], ignore_index=True)

# 같은 파일은 중복 제거
df = df.drop_duplicates(subset=["file_path"])

# CSV 저장
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("CSV 업데이트 완료")
print("전체 데이터 수:", len(df))
print(df["vuln_type"].value_counts())

display(df.tail())

Cloning into 'vulnerable-code-examples'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 184 (delta 21), reused 14 (delta 14), pack-reused 139 (from 1)
Receiving objects: 100% (184/184), 38.96 KiB | 1.62 MiB/s, done.
Resolving deltas: 100% (48/48), done.
Cloning into 'OWASP-Vulnerable-Web-Application'...
remote: Enumerating objects: 959, done.
remote: Total 959 (delta 0), reused 0 (delta 0), pack-reused 959 (from 1)
Receiving objects: 100% (959/959), 873.76 KiB | 2.95 MiB/s, done.
Resolving deltas: 100% (604/604), done.
Cloning into 'buffer-overflow-attack'...
remote: Enumerating objects: 18, done.
remote: Total 18 (delta 0), reused 0 (delta 0), pack-reused 18 (from 1)
Receiving objects: 100% (18/18), 6.56 KiB | 6.56 MiB/s, done.
Resolving deltas: 100% (4/4), done.
Cloning into 'Buffer-Overflow-Vulnerability-Lab'...
remote: Enumerating objects: 12, done.
remote: Total 12 (delta 0), re

,code,vuln_type,source,file_path
0,<?php\n\n// Cross-Site Scripting (XSS)\n$name ...,SQL_INJECTION,dehvCurtis/vulnerable-code-examples,vulnerable-code-examples/SAST/php/basic-collec...
1,"<!-- User-provided data, such as URL parameter...",SQL_INJECTION,dehvCurtis/vulnerable-code-examples,vulnerable-code-examples/SAST/php/reflection-i...
2,<?php\n\nif (PHP_SAPI === 'cli') {\n parse_...,SQL_INJECTION,dehvCurtis/vulnerable-code-examples,vulnerable-code-examples/SAST/php/sql-injectio...
3,// The Element.innerHTML property is used to r...,XSS,dehvCurtis/vulnerable-code-examples,vulnerable-code-examples/SAST/typescript/dom-x...
4,char array[10];\ninitialize(array);\nvoid *pos...,BUFFER_OVERFLOW,dehvCurtis/vulnerable-code-examples,vulnerable-code-examples/SAST/cpp/posix-buffer...


기존 CSV 데이터 수: 34
CSV 업데이트 완료
전체 데이터 수: 41
vuln_type
SQL_INJECTION        13
BUFFER_OVERFLOW       7
XSS                   7
COMMAND_INJECTION     7
PATH_TRAVERSAL        7
Name: count, dtype: int64


,code,vuln_type,source,file_path
70,/* stack.c */\n/* This program has a buffer ov...,BUFFER_OVERFLOW,npapernot/buffer-overflow-attack,buffer-overflow-attack/stack.c
71,/* call_shellcode.c */\n/*A program that creat...,BUFFER_OVERFLOW,npapernot/buffer-overflow-attack,buffer-overflow-attack/call_shellcode.c
72,/* exploit.c */\n\n/* A program that creates ...,BUFFER_OVERFLOW,wadejason/Buffer-Overflow-Vulnerability-Lab,Buffer-Overflow-Vulnerability-Lab/exploit.c
73,/* stack.c */\n\n/* This program has a buffer ...,BUFFER_OVERFLOW,wadejason/Buffer-Overflow-Vulnerability-Lab,Buffer-Overflow-Vulnerability-Lab/stack.c
74,/* call_shellcode.c */\n\n/*A program that cr...,BUFFER_OVERFLOW,wadejason/Buffer-Overflow-Vulnerability-Lab,Buffer-Overflow-Vulnerability-Lab/call_shellco...


In [31]:
from google.colab import files
files.download("github_vulnerability_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

3.기본 설정

In [11]:
import pandas as pd
import numpy as np
import torch
import json

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

DATA_PATH = "github_vulnerability_dataset.csv"
MODEL_NAME = "microsoft/codebert-base"
OUTPUT_DIR = "./codebert_vuln_type_model"

MAX_LENGTH = 256

print("설정 완료")

설정 완료


4.데이터 불러오기 및 라벨 변환

In [18]:
#csv 파일 불러옴
df = pd.read_csv(DATA_PATH)

print("전체 데이터 수:", len(df))
print("컬럼 목록:", df.columns.tolist())

if "code" not in df.columns or "vuln_type" not in df.columns:
    raise ValueError("CSV 파일에는 반드시 'code', 'vuln_type' 컬럼이 있어야 합니다.")
#결측치 제거
df = df.dropna(subset=["code", "vuln_type"])

#문자열 변환
df["code"] = df["code"].astype(str)
df["vuln_type"] = df["vuln_type"].astype(str)

#취약점 종류별 데이터 개수 확인
print("\n취약점 유형 분포")
print(df["vuln_type"].value_counts())

#취약점 종류 목록 추출
label_list = sorted(df["vuln_type"].unique())

#문자열->숫자 변환
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

df["label"] = df["vuln_type"].map(label2id)

NUM_LABELS = len(label_list)

print("\n라벨 매핑")
print(label2id)

display(df.head())

전체 데이터 수: 41
컬럼 목록: ['code', 'vuln_type', 'source', 'file_path']

취약점 유형 분포
vuln_type
SQL_INJECTION        13
BUFFER_OVERFLOW       7
XSS                   7
COMMAND_INJECTION     7
PATH_TRAVERSAL        7
Name: count, dtype: int64

라벨 매핑
{'BUFFER_OVERFLOW': 0, 'COMMAND_INJECTION': 1, 'PATH_TRAVERSAL': 2, 'SQL_INJECTION': 3, 'XSS': 4}


,code,vuln_type,source,file_path,label
0,char array[10];\ninitialize(array);\nvoid *pos...,BUFFER_OVERFLOW,dehvCurtis/vulnerable-code-examples,vulnerable-code-examples/SAST/cpp/posix-buffer...,0
1,<?php\n\n// Cross-Site Scripting (XSS)\n$name ...,SQL_INJECTION,dehvCurtis/vulnerable-code-examples,vulnerable-code-examples/SAST/php/basic-collec...,3
2,"<!-- User-provided data, such as URL parameter...",SQL_INJECTION,dehvCurtis/vulnerable-code-examples,vulnerable-code-examples/SAST/php/reflection-i...,3
3,<?php\n\nif (PHP_SAPI === 'cli') {\n parse_...,SQL_INJECTION,dehvCurtis/vulnerable-code-examples,vulnerable-code-examples/SAST/php/sql-injectio...,3
4,// The Element.innerHTML property is used to r...,XSS,dehvCurtis/vulnerable-code-examples,vulnerable-code-examples/SAST/typescript/dom-x...,4


5. 학습/검증 데이터 분리 및 토큰화

In [19]:
#전체 데이터 학습용(train),검증용(validation)으로 나누기
train_df, valid_df = train_test_split(
    df,
    test_size=0.2, #데이터 20퍼를 검증용으로 사용
    random_state=42,
    stratify=df["label"]  #특정 취약점만 너무 많아지는 것 방지
)

print("학습 데이터 수:", len(train_df))
print("검증 데이터 수:", len(valid_df))

# pandas dataframe huggingface dataset형태로 변환
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

# codebert 전용 tokenizer 불러오기
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#코드 AI입력 형태로 변환하는 함수
def tokenize_function(batch):
    return tokenizer(
        batch["code"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

#모든 코드 데이터에 tokenizer 적용
train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)

#학습에 필요 없는 컬럼 제거
remove_cols = [
    col for col in train_dataset.column_names
    if col not in ["input_ids", "attention_mask", "label"]
]

#Pytorch Tensor 형태로 변환 codeBERT는 Pytorch기반 모델이라 Tensor 데이터 형태여야지 학습이 가능
train_dataset = train_dataset.remove_columns(remove_cols)
valid_dataset = valid_dataset.remove_columns(remove_cols)

train_dataset.set_format("torch")
valid_dataset.set_format("torch")

print("토큰화 완료")

학습 데이터 수: 32
검증 데이터 수: 9


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

토큰화 완료


6.CodeBERT 모델 생성 및 학습

In [29]:
#CodeBERT 기본 지식 +취약점 데이터셋= 취약점 분류 모델 :기존 모델 가져와서 취약점 유형 분류 모델로 바꾸는 부분
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

#모델이 얼마나 잘 맞 췄는지 평가하는 함수
def compute_metrics(eval_pred):
    # 모델 예측값과 실제 정답 라벨 분리
    logits, labels = eval_pred

    # logits 중 가장 큰 값을 가진 클래스를 예측 결과로 선택
    preds = np.argmax(logits, axis=-1)


    # precision, recall, f1 계산 precision:취약하다고 예측 한 것중 실제로 맞은 비율, recall:실제 취약 코드 중 모델이 찾아낸 비율, f1:precision과 recall의 균형 점수
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    # accuracy 계산,accuracy:전체 중 맞춘 비율
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

#모델 어떻게 학습시킬 지 정하는 설정 값
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=8, #한 번에 학습할 데이터 개수, GPU 메모리가 부족하면 4로 줄이면 됨
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,

    logging_steps=10,  #10 step마다 학습 로그 출력
    logging_dir="./logs",

    load_best_model_at_end=True,
    metric_for_best_model="f1", #가장 좋은 모델 고르는 기준, f1점수는 높을수록 좋은 지표
    greater_is_better=True,

    report_to="none" #외부 로깅 도구 사용 안 함
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(f"{OUTPUT_DIR}/label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "label2id": label2id,
            "id2label": id2label
        },
        f,
        ensure_ascii=False,
        indent=4
    )

print("모델 학습 및 저장 완료")
print("저장 위치:", OUTPUT_DIR)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,1.557626,0.777778,0.777778,0.777778,0.748148
2,No log,1.522393,0.666667,0.500000,0.666667,0.555556
3,1.591977,1.505949,0.333333,0.111111,0.333333,0.166667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

모델 학습 및 저장 완료
저장 위치: ./codebert_vuln_type_model


7. 학습된 모델로 예측 테스트

In [30]:
MODEL_PATH = "./codebert_vuln_type_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# 저장된 CodeBERT 취약점 분류 모델 불러오기
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

with open(f"{MODEL_PATH}/label_mapping.json", "r", encoding="utf-8") as f:
    label_map = json.load(f)

id2label = {int(k): v for k, v in label_map["id2label"].items()}

def predict_vulnerability_type(code):
  #GPU사용 가능하면 GPU사용,안되면 CPU 사용
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)
    model.eval()
#입력 코드 토큰화(새로 넣은 데이터 한개만 토큰화)
    inputs = tokenizer(
        code,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=256
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
#예측할 때는 기울기 계산x
    with torch.no_grad():
        outputs = model(**inputs)
        #모델 원시 출력 값인 logits를 확률로 변환
        probs = torch.softmax(outputs.logits, dim=1)
    #가장 높은 확률 가진 취약점 선택
    pred_id = torch.argmax(probs, dim=1).item()
    pred_label = id2label[pred_id]
    confidence = probs[0][pred_id].item()

    return {
        "prediction": pred_label,
        "confidence": round(confidence, 4),
        "is_vulnerable": pred_label != "SAFE"
    }

test_code = """
import os

user_input = input("IP 입력: ")
os.system("ping " + user_input)
"""

result = predict_vulnerability_type(test_code)
print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'prediction': 'COMMAND_INJECTION', 'confidence': 0.207, 'is_vulnerable': True}
